In [ ]:
# ============================================================
# AUDIT DE CALIBRATION DES LLM EN MATCHING D'ESSAIS CLINIQUES
# Mémoire de Master — Régis Likassi
# Dir. Dr. Anuradha Kar (Aivancity)
# ============================================================
# DESIGN : double prompt par modèle pour isoler l'effet du prompt
#   Prompt A — Générique (minimaliste, usage naïf)
#   Prompt B — Adapté (définitions, exemples, règles d'expertise)
#
# Permet de tester :
#   - H2 : effet RLHF (Base vs Instruct, prompt fixé)
#   - Effet prompt (A vs B, modèle fixé)
#   - Interaction RLHF × prompt
# ============================================================


# ============================================================
# CELLULE 0 — Renommer les anciens fichiers en "_B" (à lancer EN PREMIER si tu reprends une session existante)
# ============================================================

In [ ]:
import os
import shutil

OUTPUT = "/kaggle/working/results"
os.makedirs(OUTPUT, exist_ok=True)

# Renommer les fichiers existants pour clarifier qu'ils sont en "prompt B (adapté)"
for nom in ['mistral_instruct', 'mistral_base', 'llama_instruct', 'llama_base']:
    for ext in ['parquet', 'csv']:
        src = f"{OUTPUT}/{nom}_predictions.{ext}"
        dst = f"{OUTPUT}/{nom}_B_predictions.{ext}"
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.move(src, dst)
            print(f"✓ {nom}_predictions.{ext} → {nom}_B_predictions.{ext}")
        elif os.path.exists(dst):
            print(f"  {nom}_B_predictions.{ext} déjà présent, rien à faire")

print("\n✓ Fichiers existants renommés en _B (prompt adapté)")


# ============================================================
# PARTIE 0 — CONFIGURATION ET IMPORTS
# ============================================================

In [ ]:
import os
import json
import time
import gc
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import (
    cohen_kappa_score, accuracy_score, f1_score,
    confusion_matrix, brier_score_loss
)
from scipy.stats import kendalltau, spearmanr
import matplotlib.pyplot as plt

BASE = "/kaggle/input/datasets/rgislikassi/llm-clinical-trial-matching-calibration-benchmark/kaggle_upload"
OUTPUT = "/kaggle/working/results"
os.makedirs(OUTPUT, exist_ok=True)

print("Configuration OK")
print(f"Dataset : {BASE}")
print(f"Sorties : {OUTPUT}")


# ============================================================
# PARTIE 1 — CHARGEMENT DES DONNÉES
# ============================================================

In [ ]:
# 1.1 — Vérifier que le dataset est attaché
if not os.path.exists(BASE):
    print("⚠ Chemin non trouvé. Contenu de /kaggle/input/ :\n")
    for root, dirs, files in os.walk("/kaggle/input/"):
        print(root)
else:
    print(f"✓ Dataset trouvé : {BASE}")

In [ ]:
# 1.2 — Charger le gold standard
df = pd.read_parquet(f"{BASE}/trialgpt_criterion/annotations.parquet")
print(f"Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"Colonnes : {list(df.columns)}")
df.head()

In [ ]:
# 1.3 — Distributions et kappa de Cohen (vérification d'intégrité)
print("=== expert_eligibility ===")
print(df['expert_eligibility'].value_counts(), "\n")

print("=== gpt4_eligibility ===")
print(df['gpt4_eligibility'].value_counts(), "\n")

kappa = cohen_kappa_score(df['expert_eligibility'], df['gpt4_eligibility'])
print(f"Kappa de Cohen expert vs GPT-4 : {kappa:.3f}")
print("→ Attendu : 0,816 (si conforme, les données sont intactes)")


# ============================================================
# PARTIE 2 — AUDIT PRÉ-LLM (DÉJÀ FAIT, à relancer si besoin)
# ============================================================

In [ ]:
# 2.1 — Mapping ordinal
ordinal_map = {
    'excluded': 0,
    'not included': 0,
    'not enough information': 1,
    'not applicable': 1,
    'not excluded': 2,
    'included': 2,
}

df['expert_score'] = df['expert_eligibility'].map(ordinal_map)
df['gpt4_score'] = df['gpt4_eligibility'].map(ordinal_map)

# Audit complet
acc_multi = (df['expert_eligibility'] == df['gpt4_eligibility']).mean()
kappa_multi = cohen_kappa_score(df['expert_eligibility'], df['gpt4_eligibility'])
kappa_ordinal = cohen_kappa_score(df['expert_score'], df['gpt4_score'], weights='quadratic')
tau, p_tau = kendalltau(df['expert_score'], df['gpt4_score'])
rho, p_rho = spearmanr(df['expert_score'], df['gpt4_score'])

print(f"Accord brut (6 labels)        : {acc_multi:.3f}")
print(f"Kappa multi-classes           : {kappa_multi:.3f}")
print(f"Kappa quadratique ordinal     : {kappa_ordinal:.3f}")
print(f"Tau de Kendall                : {tau:.3f} (p={p_tau:.2e})")
print(f"Spearman rho                  : {rho:.3f} (p={p_rho:.2e})")


# ============================================================
# PARTIE 3 — GÉNÉRATION DE SCORES AVEC LES 6 MODÈLES × 2 PROMPTS
# ============================================================

In [ ]:
# 3.0.a — Installation
!pip install -q bitsandbytes accelerate
print("Installation terminée")

In [ ]:
# 3.0.b — Authentification HuggingFace
from kaggle_secrets import UserSecretsClient

try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=hf_token)
    print("✓ Connecté à HuggingFace via Secrets")
except Exception as e:
    print(f"Secret non trouvé : {e}")

In [ ]:
# 3.0.c — Fonctions génériques pour le chargement et le scoring
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LABELS_6 = [
    "included", "not included", "excluded",
    "not excluded", "not enough information", "not applicable",
]

def liberer_gpu():
    """Libère la mémoire GPU du modèle précédent."""
    global model, tokenizer
    if 'model' in globals() and model is not None:
        del model
    if 'tokenizer' in globals() and tokenizer is not None:
        del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Mémoire GPU après libération : {torch.cuda.memory_allocated()/1e9:.2f} Go")

def charger_modele(model_id):
    """Charge n'importe quel modèle en 4-bit."""
    global model, tokenizer

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.eval()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"✓ {model_id} chargé en 4-bit")
    print(f"  Mémoire GPU utilisée : {torch.cuda.memory_allocated()/1e9:.1f} Go")
    return model, tokenizer

def est_instruct(model_id):
    """Détecte si un modèle est instruct/chat ou base."""
    mid = model_id.lower()
    return any(x in mid for x in ['instruct', '-it', 'chat'])

def score_label_sequence(prompt_ids, label_text):
    """Calcule la log-probabilité moyenne d'une séquence de tokens."""
    label_ids = tokenizer.encode(" " + label_text, add_special_tokens=False)
    full_ids = torch.cat([prompt_ids, torch.tensor([label_ids], device=model.device)], dim=1)
    with torch.no_grad():
        outputs = model(full_ids)
        logits = outputs.logits[0]
    log_probs = F.log_softmax(logits, dim=-1)
    start = prompt_ids.shape[1]
    total_lp = sum(log_probs[start + i - 1, tok].item() for i, tok in enumerate(label_ids))
    return total_lp / len(label_ids)

print("✓ Fonctions de chargement et scoring prêtes")

In [ ]:
# 3.0.d — PROMPT A (générique) pour modèles INSTRUCT
def juger_critere_6_instruct_A(note, criterion, criterion_type):
    """Version A (générique) pour modèles instruct."""

    prompt = f"""Classify whether this patient meets the eligibility criterion.

Patient: {note[:1500]}
Criterion ({criterion_type}): {criterion}

Choose exactly one label from: included, not included, excluded, not excluded, not enough information, not applicable.

Answer:"""

    messages = [{"role": "user", "content": prompt}]
    enc = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True, return_dict=True
    )
    prompt_ids = enc["input_ids"].to(model.device)

    if criterion_type == "inclusion":
        candidats = ["included", "not included", "not enough information", "not applicable"]
    else:
        candidats = ["excluded", "not excluded", "not enough information", "not applicable"]

    label_scores = {lab: score_label_sequence(prompt_ids, lab) for lab in candidats}
    scores_tensor = torch.tensor(list(label_scores.values()))
    probs = F.softmax(scores_tensor, dim=0)
    label_probs = {lab: probs[i].item() for i, lab in enumerate(candidats)}

    pred = max(label_probs, key=label_probs.get)
    confidence = label_probs[pred]
    return pred, confidence, label_probs

print("✓ juger_critere_6_instruct_A (prompt générique) prêt")

In [ ]:
# 3.0.e — PROMPT B (adapté) pour modèles INSTRUCT
def juger_critere_6_instruct_B(note, criterion, criterion_type):
    """Version B (adaptée) pour modèles instruct."""

    if criterion_type == "inclusion":
        guidance = """You are evaluating an INCLUSION criterion. To join the trial, the patient SHOULD satisfy it.

Choose exactly ONE label:

- "included" — The patient note contains information showing the patient MEETS this inclusion criterion.
  Example: criterion "Type 2 diabetes", note says "patient has T2DM" → included.

- "not included" — The patient note contains information showing the patient does NOT meet this inclusion criterion.
  Example: criterion "age over 65", note says "45-year-old patient" → not included.

- "not enough information" — The note does NOT mention anything that lets you decide. The information is simply missing.
  Example: criterion "HbA1c between 7 and 10", note never mentions HbA1c → not enough information.

- "not applicable" — The criterion concerns a different population or context and cannot logically apply to this patient.
  Example: criterion "for pediatric patients only", note describes an adult → not applicable."""

    else:  # exclusion
        guidance = """You are evaluating an EXCLUSION criterion. To join the trial, the patient should NOT have the condition described.

Choose exactly ONE label:

- "excluded" — The note shows the patient HAS the excluding condition. They ARE excluded.
  Example: criterion "pregnancy", note says "patient is pregnant" → excluded.

- "not excluded" — The patient does NOT have the excluding condition, so they can join (FAVORABLE).
  This includes TWO cases:
  (a) The note explicitly says the patient does not have it ("no history of cancer").
  (b) The note describes the patient's relevant conditions but does NOT mention this one — for a specific medical condition, absence of mention means the patient does not have it.
  Example: criterion "history of stroke", note details the patient's history with no stroke mentioned → not excluded.

- "not enough information" — Use this RARELY, only when the criterion needs a specific measurement or detail that the note genuinely cannot address and that cannot be assumed absent.
  Example: criterion "ejection fraction below 40%", note never measures ejection fraction → not enough information.

- "not applicable" — The criterion targets a different population and cannot logically apply.
  Example: criterion "pregnant women excluded", patient is male → not applicable.

CRUCIAL RULE FOR EXCLUSION CRITERIA:
If the criterion names a specific disease, condition, or medical history, and the note does NOT mention it, the default is "not excluded" (the patient most likely does not have it). Reserve "not enough information" only for criteria requiring a specific lab value or measurement that is truly missing."""

    prompt = f"""You are an expert clinical trial coordinator. Classify whether a patient meets one eligibility criterion, based only on the patient note.

{guidance}

KEY DISTINCTION:
- Use "not enough information" ONLY when the note is silent and you cannot reasonably assume the answer.
- If the note gives ANY evidence (even indirect) about the patient's status, use a decisive label (included/not included or excluded/not excluded).

---
PATIENT NOTE:
{note[:1500]}

CRITERION ({criterion_type}):
{criterion}
---

Reason about whether the note addresses this criterion, then give your final label.
Final label:"""

    messages = [{"role": "user", "content": prompt}]
    enc = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True, return_dict=True
    )
    prompt_ids = enc["input_ids"].to(model.device)

    if criterion_type == "inclusion":
        candidats = ["included", "not included", "not enough information", "not applicable"]
    else:
        candidats = ["excluded", "not excluded", "not enough information", "not applicable"]

    label_scores = {lab: score_label_sequence(prompt_ids, lab) for lab in candidats}
    scores_tensor = torch.tensor(list(label_scores.values()))
    probs = F.softmax(scores_tensor, dim=0)
    label_probs = {lab: probs[i].item() for i, lab in enumerate(candidats)}

    pred = max(label_probs, key=label_probs.get)
    confidence = label_probs[pred]
    return pred, confidence, label_probs

print("✓ juger_critere_6_instruct_B (prompt adapté) prêt")

In [ ]:
# 3.0.f — PROMPT A (générique) pour modèles BASE
def juger_critere_6_base_A(note, criterion, criterion_type):
    """Version A (générique) pour modèles base. Un seul exemple court."""

    few_shot = """Example:
Patient note: 65-year-old male with type 2 diabetes.
Criterion (inclusion): diabetes
Final label: included

"""

    if criterion_type == "inclusion":
        candidats = ["included", "not included", "not enough information", "not applicable"]
    else:
        candidats = ["excluded", "not excluded", "not enough information", "not applicable"]

    prompt = f"""Task: Classify whether a patient meets one eligibility criterion. Choose exactly one label from: included, not included, excluded, not excluded, not enough information, not applicable.

{few_shot}Now classify:
Patient note: {note[:1500]}
Criterion ({criterion_type}): {criterion}
Final label:"""

    prompt_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

    label_scores = {lab: score_label_sequence(prompt_ids, lab) for lab in candidats}
    scores_tensor = torch.tensor(list(label_scores.values()))
    probs = F.softmax(scores_tensor, dim=0)
    label_probs = {lab: probs[i].item() for i, lab in enumerate(candidats)}

    pred = max(label_probs, key=label_probs.get)
    confidence = label_probs[pred]
    return pred, confidence, label_probs

print("✓ juger_critere_6_base_A (prompt générique) prêt")

In [ ]:
# 3.0.g — PROMPT B (adapté) pour modèles BASE
def juger_critere_6_base_B(note, criterion, criterion_type):
    """Version B (adaptée) pour modèles base. Few-shot riche."""

    if criterion_type == "inclusion":
        few_shot = """Example 1:
Patient note: 65-year-old male with type 2 diabetes, HbA1c 8.5%.
Criterion (inclusion): Type 2 diabetes mellitus
Final label: included

Example 2:
Patient note: 45-year-old female, no diabetes history.
Criterion (inclusion): age over 65
Final label: not included

Example 3:
Patient note: 50-year-old patient with hypertension.
Criterion (inclusion): HbA1c between 7 and 10
Final label: not enough information

"""
        candidats = ["included", "not included", "not enough information", "not applicable"]

    else:  # exclusion
        few_shot = """Example 1:
Patient note: 30-year-old female, 12 weeks pregnant.
Criterion (exclusion): pregnancy
Final label: excluded

Example 2:
Patient note: 55-year-old male with hypertension and diabetes, no cancer history.
Criterion (exclusion): history of cancer
Final label: not excluded

Example 3:
Patient note: 60-year-old male with hypertension. Otherwise healthy.
Criterion (exclusion): history of stroke
Final label: not excluded

Example 4:
Patient note: 70-year-old male with chronic kidney disease.
Criterion (exclusion): ejection fraction below 40%
Final label: not enough information

"""
        candidats = ["excluded", "not excluded", "not enough information", "not applicable"]

    prompt = f"""Task: Classify whether a patient meets one eligibility criterion, based only on the patient note. Choose exactly one label from: included, not included, excluded, not excluded, not enough information, not applicable.

{few_shot}Now classify this case:
Patient note: {note[:1500]}
Criterion ({criterion_type}): {criterion}
Final label:"""

    prompt_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

    label_scores = {lab: score_label_sequence(prompt_ids, lab) for lab in candidats}
    scores_tensor = torch.tensor(list(label_scores.values()))
    probs = F.softmax(scores_tensor, dim=0)
    label_probs = {lab: probs[i].item() for i, lab in enumerate(candidats)}

    pred = max(label_probs, key=label_probs.get)
    confidence = label_probs[pred]
    return pred, confidence, label_probs

print("✓ juger_critere_6_base_B (prompt adapté) prêt")

In [ ]:
# 3.0.h — Fonction générique de RUN COMPLET avec choix du prompt
from tqdm.notebook import tqdm

def run_complet(model_id, output_name, prompt_version):
    """
    Lance un run complet sur les 1015 paires.
    prompt_version : 'A' (générique) ou 'B' (adapté)
    output_name : nom de base, le suffixe _A ou _B est ajouté automatiquement
    """
    # Choix automatique de la bonne fonction
    if est_instruct(model_id):
        if prompt_version == 'A':
            juger_fn = juger_critere_6_instruct_A
        else:
            juger_fn = juger_critere_6_instruct_B
        print(f"→ Mode INSTRUCT, prompt {prompt_version}")
    else:
        if prompt_version == 'A':
            juger_fn = juger_critere_6_base_A
        else:
            juger_fn = juger_critere_6_base_B
        print(f"→ Mode BASE, prompt {prompt_version}")

    nom_complet = f"{output_name}_{prompt_version}"
    resultats = []
    t0 = time.time()

    for _, row in tqdm(df.iterrows(), total=len(df), desc=nom_complet):
        pred, conf, probs = juger_fn(row['note'], row['criterion_text'], row['criterion_type'])
        rec = {
            'annotation_id': row['annotation_id'],
            'criterion_type': row['criterion_type'],
            'pred': pred,
            'confidence': conf,
            'expert': row['expert_eligibility'],
            'gpt4': row['gpt4_eligibility'],
            'prompt_version': prompt_version,
        }
        for lab in LABELS_6:
            rec[f"p_{lab.replace(' ','_')}"] = probs.get(lab, float('nan'))
        resultats.append(rec)

    duree = time.time() - t0
    res_df = pd.DataFrame(resultats)

    res_df.to_parquet(f"{OUTPUT}/{nom_complet}_predictions.parquet", index=False)
    res_df.to_csv(f"{OUTPUT}/{nom_complet}_predictions.csv", index=False)

    accord = (res_df['pred'] == res_df['expert']).mean()
    print(f"\n=== {nom_complet} TERMINÉ ===")
    print(f"Durée : {duree/60:.1f} min ({duree/len(df):.1f}s par paire)")
    print(f"Accord {nom_complet} / expert : {accord:.1%}")
    print(f"Sauvegardé : {OUTPUT}/{nom_complet}_predictions.parquet")
    return res_df

print("✓ Fonction run_complet prête (paramètre prompt_version : 'A' ou 'B'-)")


# ============================================================
# 3.1 — MISTRAL-7B (prompt B déjà fait, ne lancer que A)
# ============================================================

In [ ]:
# 3.1.a — Mistral-7B-Instruct, Prompt A (générique)
liberer_gpu()
charger_modele("mistralai/Mistral-7B-Instruct-v0.3")
mistral_instruct_A = run_complet("mistralai/Mistral-7B-Instruct-v0.3", "mistral_instruct", "A")

In [ ]:
# 3.1.b — Mistral-7B-Base, Prompt A (générique)
liberer_gpu()
charger_modele("mistralai/Mistral-7B-v0.3")
mistral_base_A = run_complet("mistralai/Mistral-7B-v0.3", "mistral_base", "A")


# ============================================================
# 3.2 — LLAMA-3.1-8B (prompt B déjà fait, ne lancer que A)
# ============================================================

In [ ]:
# 3.2.a — Llama-3.1-8B-Instruct, Prompt A (générique)
liberer_gpu()
charger_modele("meta-llama/Llama-3.1-8B-Instruct")
llama_instruct_A = run_complet("meta-llama/Llama-3.1-8B-Instruct", "llama_instruct", "A")

In [ ]:
# 3.2.b — Llama-3.1-8B-Base, Prompt A (générique)
liberer_gpu()
charger_modele("meta-llama/Llama-3.1-8B")
llama_base_A = run_complet("meta-llama/Llama-3.1-8B", "llama_base", "A")


# ============================================================
# 3.3 — MEDITRON-7B (deux prompts à lancer)
# ============================================================

In [ ]:
# 3.3.a — Meditron-7B, Prompt A (générique)
liberer_gpu()
charger_modele("epfl-llm/meditron-7b")
meditron_A = run_complet("epfl-llm/meditron-7b", "meditron", "A")

In [ ]:
# 3.3.b — Meditron-7B, Prompt B (adapté)
# Le modèle est déjà chargé, pas besoin de relibérer
meditron_B = run_complet("epfl-llm/meditron-7b", "meditron", "B")


# ============================================================
# 3.4 — BIOMISTRAL-7B (deux prompts à lancer)
# ============================================================

In [ ]:
# 3.4.a — BioMistral-7B, Prompt A (générique)
liberer_gpu()
charger_modele("BioMistral/BioMistral-7B")
biomistral_A = run_complet("BioMistral/BioMistral-7B", "biomistral", "A")

In [ ]:
# 3.4.b — BioMistral-7B, Prompt B (adapté)
biomistral_B = run_complet("BioMistral/BioMistral-7B", "biomistral", "B")


# ============================================================
# PARTIE 4 — AUDIT DE CALIBRATION (12 audits : 6 modèles × 2 prompts)
# ============================================================

In [ ]:
# 4.1 — Fonctions de calibration
def expected_calibration_error(y_true, y_prob, n_bins=10):
    """ECE avec binning uniforme."""
    bins = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    ece = 0.0
    bin_data = []
    for k in range(n_bins):
        mask = (bin_ids == k)
        if mask.sum() > 0:
            acc_k = y_true[mask].mean()
            conf_k = y_prob[mask].mean()
            weight = mask.sum() / len(y_true)
            ece += weight * abs(acc_k - conf_k)
            bin_data.append({
                'bin': k, 'low': bins[k], 'high': bins[k+1],
                'count': int(mask.sum()),
                'accuracy': float(acc_k),
                'confidence': float(conf_k),
                'gap': float(abs(acc_k - conf_k)),
            })
    return ece, pd.DataFrame(bin_data)

def bootstrap_ece_ci(y_true, y_prob, n_bins=10, n_boot=1000, seed=42):
    """IC 95% sur l'ECE par bootstrap."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    eces = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        e, _ = expected_calibration_error(y_true[idx], y_prob[idx], n_bins)
        eces.append(e)
    return np.percentile(eces, 2.5), np.percentile(eces, 97.5)

def audit_calibration(predictions_df, model_name, output_dir=None):
    """Audit complet d'un modèle."""
    df_pred = predictions_df.copy()
    df_pred['correct'] = (df_pred['pred'] == df_pred['expert']).astype(int)
    y_true = df_pred['correct'].values
    y_prob = df_pred['confidence'].values

    accord = df_pred['correct'].mean()
    ece, bin_df = expected_calibration_error(y_true, y_prob, n_bins=10)
    ece_low, ece_high = bootstrap_ece_ci(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob)
    h1_supportee = ece_low > 0.05

    print(f"\n{'='*60}")
    print(f" CALIBRATION — {model_name}")
    print(f"{'='*60}")
    print(f"  N paires                    : {len(df_pred)}")
    print(f"  Accord (accuracy)           : {accord:.3f}")
    print(f"  Confiance moyenne           : {y_prob.mean():.3f}")
    print(f"  Sur-confiance moyenne       : {(y_prob - y_true).mean():+.3f}")
    print(f"  ECE (10 bins)               : {ece:.4f}")
    print(f"  ECE 95% IC                  : [{ece_low:.4f} ; {ece_high:.4f}]")
    print(f"  Brier score                 : {brier:.4f}")
    print(f"  ★ H1 (ECE > 0.05)           : {'SUPPORTÉE → mal calibré' if h1_supportee else 'NON supportée'}")

    results = {
        'model': model_name, 'n': int(len(df_pred)),
        'accuracy': float(accord), 'ece': float(ece),
        'ece_ci_low': float(ece_low), 'ece_ci_high': float(ece_high),
        'brier': float(brier),
        'mean_confidence': float(y_prob.mean()),
        'mean_overconfidence': float((y_prob - y_true).mean()),
        'h1_supportee': bool(h1_supportee),
    }
    if output_dir:
        safe = model_name.replace('/', '_').replace(' ', '_')
        with open(f"{output_dir}/calib_{safe}.json", 'w') as f:
            json.dump(results, f, indent=2)
        bin_df.to_csv(f"{output_dir}/calib_{safe}_bins.csv", index=False)
    return results, bin_df

def reliability_diagram(bin_df, model_name, output_dir=None, ax=None):
    """Trace le reliability diagram."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 7))

    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Calibration parfaite')
    ax.plot(bin_df['confidence'], bin_df['accuracy'], 'o-',
            markersize=10, linewidth=2, label=model_name)

    ax2 = ax.twinx()
    ax2.bar(bin_df['confidence'], bin_df['count'],
            width=0.08, alpha=0.2, color='gray')
    ax2.set_ylabel('Nb prédictions (gris)', color='gray')
    ax2.tick_params(axis='y', labelcolor='gray')

    ax.set_xlabel('Confiance prédite', fontsize=12)
    ax.set_ylabel('Précision observée', fontsize=12)
    ax.set_title(f'Reliability Diagram — {model_name}', fontsize=13)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(loc='upper left')
    ax.grid(alpha=0.3)

    if output_dir:
        safe = model_name.replace('/', '_').replace(' ', '_')
        plt.tight_layout()
        plt.savefig(f"{output_dir}/reliability_{safe}.png", dpi=150, bbox_inches='tight')
    return ax

print("✓ Fonctions de calibration prêtes")

In [ ]:
# 4.2 — Audit BATCH de tous les modèles existants (cherche automatiquement les .parquet)
import glob

tous_resultats = []
tous_bins = {}

# Lister tous les fichiers de prédictions
fichiers = sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet"))
print(f"Fichiers de prédictions trouvés : {len(fichiers)}")
for f in fichiers:
    print(f"  - {os.path.basename(f)}")

for fichier in fichiers:
    # Extraire le nom du modèle depuis le nom de fichier
    nom = os.path.basename(fichier).replace('_predictions.parquet', '')
    df_pred = pd.read_parquet(fichier)
    res, bins = audit_calibration(df_pred, nom, OUTPUT)
    tous_resultats.append(res)
    tous_bins[nom] = bins

In [ ]:
# 4.3 — Tableau récapitulatif
recap = pd.DataFrame(tous_resultats)
recap = recap[[
    'model', 'n', 'accuracy', 'ece', 'ece_ci_low', 'ece_ci_high',
    'brier', 'mean_confidence', 'mean_overconfidence', 'h1_supportee'
]]
print("\n=== TABLEAU RÉCAPITULATIF DE TOUS LES MODÈLES ===\n")
print(recap.to_string(index=False))
recap.to_csv(f"{OUTPUT}/recap_calibration_tous_modeles.csv", index=False)


# ============================================================
# PARTIE 5 — TESTS D'HYPOTHÈSES SCIENTIFIQUES
# ============================================================
# Q1 (H2)   : Effet RLHF — Base vs Instruct, pour chaque prompt
# Q2 (NEW)  : Effet du prompt — A vs B, pour chaque modèle
# Q3 (NEW)  : Interaction RLHF × prompt — l'effet RLHF dépend-il du prompt ?
# Q4 (H5)   : Effet spécialisation médicale
# ============================================================

In [ ]:
# 5.1 — Fonctions de test de permutation
def permutation_test_ece(df_a, df_b, n_perm=1000, seed=42):
    """Test de permutation unilatéral : ECE(B) > ECE(A) ?"""
    rng = np.random.default_rng(seed)

    df_a = df_a.copy()
    df_b = df_b.copy()
    df_a['correct'] = (df_a['pred'] == df_a['expert']).astype(int)
    df_b['correct'] = (df_b['pred'] == df_b['expert']).astype(int)

    ece_a, _ = expected_calibration_error(df_a['correct'].values, df_a['confidence'].values)
    ece_b, _ = expected_calibration_error(df_b['correct'].values, df_b['confidence'].values)
    diff_obs = ece_b - ece_a

    pooled_correct = np.concatenate([df_a['correct'].values, df_b['correct'].values])
    pooled_conf = np.concatenate([df_a['confidence'].values, df_b['confidence'].values])
    n_a = len(df_a)

    diffs_perm = []
    for _ in range(n_perm):
        idx = rng.permutation(len(pooled_correct))
        a_idx, b_idx = idx[:n_a], idx[n_a:]
        e_a, _ = expected_calibration_error(pooled_correct[a_idx], pooled_conf[a_idx])
        e_b, _ = expected_calibration_error(pooled_correct[b_idx], pooled_conf[b_idx])
        diffs_perm.append(e_b - e_a)

    p_value = (np.array(diffs_perm) >= diff_obs).mean()
    return diff_obs, p_value, ece_a, ece_b

def tester_comparaison(df_a, df_b, nom_a, nom_b, hypothese, alpha=0.05/8):
    """Compare deux modèles. Bonferroni sur 8 comparaisons."""
    diff, p, ece_a, ece_b = permutation_test_ece(df_a, df_b)

    print(f"\n=== {nom_a} vs {nom_b} ({hypothese}) ===")
    print(f"  ECE {nom_a:30} : {ece_a:.4f}")
    print(f"  ECE {nom_b:30} : {ece_b:.4f}")
    print(f"  Δ ({nom_b} - {nom_a}) : {diff:+.4f}")
    print(f"  p-value (perm)             : {p:.4f}")
    print(f"  Seuil α corrigé Bonferroni : {alpha:.4f}")
    supportee = (p < alpha) and (diff > 0)
    print(f"  ★ {hypothese} supportée ?   : {'OUI ✓' if supportee else 'NON'}")
    return {
        'comparaison': f"{nom_a} vs {nom_b}",
        'hypothese': hypothese,
        'ece_a': ece_a, 'ece_b': ece_b,
        'delta': diff, 'p_value': p,
        'supportee': supportee
    }

# Helper pour charger les prédictions
def charger(nom):
    """Charge un fichier de prédictions par son nom de base."""
    chemin = f"{OUTPUT}/{nom}_predictions.parquet"
    if os.path.exists(chemin):
        return pd.read_parquet(chemin)
    else:
        print(f"⚠ Fichier manquant : {chemin}")
        return None

print("✓ Fonctions de tests d'hypothèses prêtes")

In [ ]:
# 5.2 — Q1 (H2) : Effet du RLHF, à prompt fixé
print("="*70)
print(" Q1 (H2) — Le RLHF dégrade-t-il la calibration ?")
print("="*70)

tests_h2 = []

# Mistral, prompt A
mi_A = charger("mistral_instruct_A")
mb_A = charger("mistral_base_A")
if mi_A is not None and mb_A is not None:
    tests_h2.append(tester_comparaison(
        mb_A, mi_A, "Mistral-Base (A)", "Mistral-Instruct (A)",
        "H2 (Mistral, prompt A)"
    ))

# Mistral, prompt B
mi_B = charger("mistral_instruct_B")
mb_B = charger("mistral_base_B")
if mi_B is not None and mb_B is not None:
    tests_h2.append(tester_comparaison(
        mb_B, mi_B, "Mistral-Base (B)", "Mistral-Instruct (B)",
        "H2 (Mistral, prompt B)"
    ))

# Llama, prompt A
li_A = charger("llama_instruct_A")
lb_A = charger("llama_base_A")
if li_A is not None and lb_A is not None:
    tests_h2.append(tester_comparaison(
        lb_A, li_A, "Llama-Base (A)", "Llama-Instruct (A)",
        "H2 (Llama, prompt A)"
    ))

# Llama, prompt B
li_B = charger("llama_instruct_B")
lb_B = charger("llama_base_B")
if li_B is not None and lb_B is not None:
    tests_h2.append(tester_comparaison(
        lb_B, li_B, "Llama-Base (B)", "Llama-Instruct (B)",
        "H2 (Llama, prompt B)"
    ))

In [ ]:
# 5.3 — Q2 (NEW) : Effet du prompt, à modèle fixé
print("="*70)
print(" Q2 (NEW) — Le prompt adapté améliore-t-il la calibration ?")
print("="*70)

tests_prompt = []

for nom in ['mistral_instruct', 'mistral_base', 'llama_instruct', 'llama_base',
            'meditron', 'biomistral']:
    df_A = charger(f"{nom}_A")
    df_B = charger(f"{nom}_B")
    if df_A is not None and df_B is not None:
        # Hypothèse : Prompt B (adapté) a un ECE plus bas que Prompt A (générique)
        # Donc on teste si ECE(A) > ECE(B) → diff positive = B améliore
        tests_prompt.append(tester_comparaison(
            df_B, df_A, f"{nom} (B)", f"{nom} (A)",
            f"Effet prompt ({nom})"
        ))

In [ ]:
# 5.4 — Q4 (H5) : Effet de la spécialisation médicale (prompt B)
print("="*70)
print(" Q4 (H5) — La spécialisation médicale améliore-t-elle la calibration ?")
print("="*70)

tests_h5 = []

# Meditron vs Llama-Base (pré-entraînement médical sans RLHF)
med_B = charger("meditron_B")
lb_B = charger("llama_base_B")
if med_B is not None and lb_B is not None:
    tests_h5.append(tester_comparaison(
        med_B, lb_B, "Meditron-7B (B)", "Llama-Base (B)",
        "H5 (vs Llama-Base)"
    ))

# BioMistral vs Mistral-Instruct (pré-entraînement médical avec RLHF hérité)
bio_B = charger("biomistral_B")
mi_B = charger("mistral_instruct_B")
if bio_B is not None and mi_B is not None:
    tests_h5.append(tester_comparaison(
        bio_B, mi_B, "BioMistral (B)", "Mistral-Instruct (B)",
        "H5 (vs Mistral-Instruct)"
    ))

In [ ]:
# 5.5 — Sauvegarde de tous les tests
tous_tests = tests_h2 + tests_prompt + tests_h5
tests_df = pd.DataFrame(tous_tests)
tests_df.to_csv(f"{OUTPUT}/tous_tests_hypotheses.csv", index=False)

print("\n=== TABLEAU GÉNÉRAL DES TESTS ===")
print(tests_df.to_string(index=False))

In [ ]:
# 5.6 — Reliability diagram comparatif final (12 modèles si tout est fait)
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Plot A : Prompt A
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Parfait')
colors = plt.cm.tab10.colors
for i, nom in enumerate(['mistral_instruct_A', 'mistral_base_A',
                          'llama_instruct_A', 'llama_base_A',
                          'meditron_A', 'biomistral_A']):
    if nom in tous_bins:
        bdf = tous_bins[nom]
        ax.plot(bdf['confidence'], bdf['accuracy'], 'o-',
                markersize=8, linewidth=2, label=nom, color=colors[i % 10])
ax.set_xlabel('Confiance prédite', fontsize=11)
ax.set_ylabel('Précision observée', fontsize=11)
ax.set_title('Prompt A (générique)', fontsize=13)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(loc='upper left', fontsize=9)
ax.grid(alpha=0.3)

# Plot B : Prompt B
ax = axes[1]
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Parfait')
for i, nom in enumerate(['mistral_instruct_B', 'mistral_base_B',
                          'llama_instruct_B', 'llama_base_B',
                          'meditron_B', 'biomistral_B']):
    if nom in tous_bins:
        bdf = tous_bins[nom]
        ax.plot(bdf['confidence'], bdf['accuracy'], 'o-',
                markersize=8, linewidth=2, label=nom, color=colors[i % 10])
ax.set_xlabel('Confiance prédite', fontsize=11)
ax.set_ylabel('Précision observée', fontsize=11)
ax.set_title('Prompt B (adapté)', fontsize=13)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(loc='upper left', fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('Calibration : Prompt A vs Prompt B sur les 6 modèles', fontsize=14)
plt.tight_layout()
plt.savefig(f"{OUTPUT}/reliability_compare_AvsB.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Figure sauvegardée : {OUTPUT}/reliability_compare_AvsB.png")
print("\n" + "="*70)
print(" FIN DE L'ANALYSE COMPARATIVE")
print("="*70)

PARTIE 6 — Validité ordinale (H3)

In [ ]:
# 6.1 — Fonction C-index ordinal sur les 6 labels
# Mappe label prédit + confiance vers un score continu, puis C-index vs expert_score
from itertools import combinations

LABEL_TO_SCORE = {
    'excluded': 0.0, 'not included': 0.0,
    'not enough information': 1.0, 'not applicable': 1.0,
    'not excluded': 2.0, 'included': 2.0,
}

def score_continu(row):
    """Score continu = score ordinal du label + (confiance × biais)."""
    base = LABEL_TO_SCORE.get(row['pred'], 1.0)
    # Plus confiant = plus de poids sur l'extrême choisi
    if base == 0.0:
        return base - row['confidence'] * 0.5  # vers −0.5
    elif base == 2.0:
        return base + row['confidence'] * 0.5  # vers 2.5
    else:
        return base  # neutre

def c_index(y_score, y_true):
    """Calcule le C-index (concordance) sur des paires ordinales."""
    pairs = 0
    concordant = 0
    tied = 0
    for i, j in combinations(range(len(y_true)), 2):
        if y_true[i] == y_true[j]:
            continue
        pairs += 1
        if y_true[i] < y_true[j]:
            if y_score[i] < y_score[j]:
                concordant += 1
            elif y_score[i] == y_score[j]:
                tied += 1
        else:
            if y_score[i] > y_score[j]:
                concordant += 1
            elif y_score[i] == y_score[j]:
                tied += 1
    return (concordant + 0.5 * tied) / pairs if pairs > 0 else 0.0

print("✓ Fonctions C-index prêtes")

In [ ]:
# 6.2 — Audit ordinal H3 pour les 12 modèles
import glob

EXPERT_TO_SCORE = LABEL_TO_SCORE

resultats_h3 = []
for fichier in sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet")):
    nom = os.path.basename(fichier).replace('_predictions.parquet', '')
    dfp = pd.read_parquet(fichier)

    # Score ordinal continu et vérité ordinale experte
    dfp['score_continu'] = dfp.apply(score_continu, axis=1)
    dfp['expert_ordinal'] = dfp['expert'].map(EXPERT_TO_SCORE)

    # C-index (sur sous-échantillon pour vitesse, sinon trop long pour 1015)
    rng = np.random.default_rng(42)
    n_sample = 500
    idx = rng.choice(len(dfp), n_sample, replace=False)
    sub = dfp.iloc[idx]
    cidx = c_index(sub['score_continu'].values, sub['expert_ordinal'].values)

    # Kendall tau et Spearman complets (plus rapides)
    tau, p_tau = kendalltau(dfp['score_continu'], dfp['expert_ordinal'])
    rho, p_rho = spearmanr(dfp['score_continu'], dfp['expert_ordinal'])

    h3_supportee = cidx > 0.70

    resultats_h3.append({
        'model': nom,
        'c_index': float(cidx),
        'kendall_tau': float(tau),
        'spearman_rho': float(rho),
        'h3_supportee': bool(h3_supportee)
    })

    print(f"{nom:25} | C-index={cidx:.3f} | tau={tau:.3f} | rho={rho:.3f} | H3 {'✓' if h3_supportee else '✗'}")

h3_df = pd.DataFrame(resultats_h3)
h3_df.to_csv(f"{OUTPUT}/h3_ordinal_audit.csv", index=False)
print(f"\n✓ Sauvegardé : {OUTPUT}/h3_ordinal_audit.csv")

PARTIE 7 — Classification sélective (H4)


In [ ]:
# 7.1 — Classification sélective : courbes risque-couverture
def courbe_risque_couverture(df_pred, n_points=20):
    """
    Pour chaque seuil de confiance, calcule :
    - couverture (% de prédictions retenues)
    - F1 macro sur les prédictions retenues
    """
    dfp = df_pred.copy()
    dfp['correct'] = (dfp['pred'] == dfp['expert']).astype(int)

    seuils = np.linspace(0, dfp['confidence'].max(), n_points)
    resultats = []
    for s in seuils:
        retenues = dfp[dfp['confidence'] >= s]
        if len(retenues) == 0:
            continue
        couverture = len(retenues) / len(dfp)
        # F1 macro
        try:
            f1 = f1_score(retenues['expert'], retenues['pred'], average='macro', zero_division=0)
        except:
            f1 = np.nan
        accuracy = retenues['correct'].mean()
        resultats.append({
            'seuil': s, 'couverture': couverture,
            'f1_macro': f1, 'accuracy': accuracy,
            'n_retenues': len(retenues)
        })
    return pd.DataFrame(resultats)

# Test sur Mistral-Instruct B pour vérifier
courbe = courbe_risque_couverture(pd.read_parquet(f"{OUTPUT}/mistral_instruct_B_predictions.parquet"))
print("Aperçu de la courbe risque-couverture (Mistral-Instruct B) :")
print(courbe.head(10).to_string(index=False))

In [ ]:
# 7.2 — Test H4 : F1(80%) vs F1(100%) sur les 12 modèles
resultats_h4 = []
for fichier in sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet")):
    nom = os.path.basename(fichier).replace('_predictions.parquet', '')
    dfp = pd.read_parquet(fichier)
    dfp = dfp.copy()
    dfp['correct'] = (dfp['pred'] == dfp['expert']).astype(int)

    # F1 à couverture 100%
    f1_full = f1_score(dfp['expert'], dfp['pred'], average='macro', zero_division=0)

    # F1 à couverture 80% (on garde les 80% les plus confiants)
    seuil_80 = dfp['confidence'].quantile(0.20)  # on retire les 20% les moins confiants
    retenues = dfp[dfp['confidence'] >= seuil_80]
    f1_80 = f1_score(retenues['expert'], retenues['pred'], average='macro', zero_division=0)

    # F1 à couverture 50% (les 50% les plus confiants)
    seuil_50 = dfp['confidence'].quantile(0.50)
    retenues50 = dfp[dfp['confidence'] >= seuil_50]
    f1_50 = f1_score(retenues50['expert'], retenues50['pred'], average='macro', zero_division=0)

    delta_80 = f1_80 - f1_full
    h4_supportee = delta_80 > 0.02

    resultats_h4.append({
        'model': nom,
        'f1_100': float(f1_full),
        'f1_80': float(f1_80),
        'f1_50': float(f1_50),
        'delta_80_vs_100': float(delta_80),
        'h4_supportee': bool(h4_supportee)
    })

    print(f"{nom:25} | F1@100={f1_full:.3f} | F1@80={f1_80:.3f} | Δ={delta_80:+.3f} | H4 {'✓' if h4_supportee else '✗'}")

h4_df = pd.DataFrame(resultats_h4)
h4_df.to_csv(f"{OUTPUT}/h4_classification_selective.csv", index=False)

In [ ]:
# 7.3 — Figure des courbes risque-couverture
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Sélection : 6 modèles représentatifs (les B pour la cohérence)
modeles_a_afficher = [
    'mistral_instruct_B', 'mistral_base_B',
    'llama_instruct_B', 'llama_base_B',
    'meditron_B', 'biomistral_B'
]

# Panel A : F1 vs couverture
ax = axes[0]
colors = plt.cm.tab10.colors
for i, nom in enumerate(modeles_a_afficher):
    fichier = f"{OUTPUT}/{nom}_predictions.parquet"
    if os.path.exists(fichier):
        dfp = pd.read_parquet(fichier)
        courbe = courbe_risque_couverture(dfp, n_points=30)
        ax.plot(courbe['couverture'], courbe['f1_macro'], 'o-',
                label=nom, color=colors[i % 10], markersize=5)
ax.set_xlabel('Couverture (% prédictions retenues)', fontsize=11)
ax.set_ylabel('F1 macro', fontsize=11)
ax.set_title('Courbes risque-couverture (F1)', fontsize=12)
ax.legend(loc='lower left', fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(0, 1.05)

# Panel B : Accuracy vs couverture
ax = axes[1]
for i, nom in enumerate(modeles_a_afficher):
    fichier = f"{OUTPUT}/{nom}_predictions.parquet"
    if os.path.exists(fichier):
        dfp = pd.read_parquet(fichier)
        courbe = courbe_risque_couverture(dfp, n_points=30)
        ax.plot(courbe['couverture'], courbe['accuracy'], 'o-',
                label=nom, color=colors[i % 10], markersize=5)
ax.set_xlabel('Couverture (% prédictions retenues)', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Courbes risque-couverture (Accuracy)', fontsize=12)
ax.legend(loc='lower left', fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(0, 1.05)

plt.suptitle("Classification sélective : performance vs couverture", fontsize=13)
plt.tight_layout()
plt.savefig(f"{OUTPUT}/courbes_risque_couverture.png", dpi=150, bbox_inches='tight')
plt.show()

PARTIE 8 — Biais "not applicable" sur les LLM open source

In [ ]:
# 8 — Biais d'esquive "not applicable" : étendu aux 6 modèles open source
# Pour chaque modèle, calculer le ratio (n_modèle/n_expert) sur "not applicable" et "not enough information"

n_expert_na = (df['expert_eligibility'] == 'not applicable').sum()
n_expert_nei = (df['expert_eligibility'] == 'not enough information').sum()

biais = []
for fichier in sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet")):
    nom = os.path.basename(fichier).replace('_predictions.parquet', '')
    dfp = pd.read_parquet(fichier)

    n_model_na = (dfp['pred'] == 'not applicable').sum()
    n_model_nei = (dfp['pred'] == 'not enough information').sum()

    ratio_na = n_model_na / n_expert_na if n_expert_na > 0 else float('inf')
    ratio_nei = n_model_nei / n_expert_nei if n_expert_nei > 0 else float('inf')

    biais.append({
        'model': nom,
        'n_NA_modele': int(n_model_na),
        'n_NA_expert': int(n_expert_na),
        'ratio_NA': float(ratio_na),
        'n_NEI_modele': int(n_model_nei),
        'n_NEI_expert': int(n_expert_nei),
        'ratio_NEI': float(ratio_nei),
    })

biais_df = pd.DataFrame(biais)
print("Biais d'esquive (ratio modèle/expert) :\n")
print(biais_df.to_string(index=False))
biais_df.to_csv(f"{OUTPUT}/biais_esquive.csv", index=False)

print("\nRappel GPT-4 : ratio NA = 2,16 | ratio NEI = 0,99")

PARTIE 9 — Stratification inclusion vs exclusion


In [ ]:
# 9 — ECE séparé pour critères d'inclusion vs exclusion
strat = []
for fichier in sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet")):
    nom = os.path.basename(fichier).replace('_predictions.parquet', '')
    dfp = pd.read_parquet(fichier)
    dfp['correct'] = (dfp['pred'] == dfp['expert']).astype(int)

    for ctype in ['inclusion', 'exclusion']:
        sub = dfp[dfp['criterion_type'] == ctype]
        if len(sub) == 0:
            continue
        ece, _ = expected_calibration_error(
            sub['correct'].values, sub['confidence'].values, n_bins=10
        )
        accord = sub['correct'].mean()
        strat.append({
            'model': nom,
            'criterion_type': ctype,
            'n': len(sub),
            'accuracy': float(accord),
            'ece': float(ece),
            'mean_confidence': float(sub['confidence'].mean()),
        })

strat_df = pd.DataFrame(strat)
print("ECE stratifié par type de critère :\n")
print(strat_df.to_string(index=False))
strat_df.to_csv(f"{OUTPUT}/ece_stratifie.csv", index=False)

PARTIE 10 — Recalibration post-hoc


In [ ]:
# 10.1 — Méthodes de recalibration
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

def temperature_scaling_fit(y_true, y_prob, n_T=200):
    """Trouve la température T qui minimise le NLL."""
    Ts = np.linspace(0.1, 5.0, n_T)
    best_T, best_nll = 1.0, np.inf
    eps = 1e-7
    for T in Ts:
        # Pour les probas calibrées à 1-class (confiance binaire correct/incorrect)
        p_T = np.clip(y_prob ** (1/T) / (y_prob ** (1/T) + (1 - y_prob) ** (1/T)), eps, 1 - eps)
        nll = -np.mean(y_true * np.log(p_T) + (1 - y_true) * np.log(1 - p_T))
        if nll < best_nll:
            best_nll, best_T = nll, T
    return best_T

def apply_temperature(y_prob, T):
    eps = 1e-7
    return np.clip(y_prob ** (1/T) / (y_prob ** (1/T) + (1 - y_prob) ** (1/T)), eps, 1 - eps)

def isotonic_calibrate(y_true_train, y_prob_train, y_prob_test):
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(y_prob_train, y_true_train)
    return iso.predict(y_prob_test)

def platt_calibrate(y_true_train, y_prob_train, y_prob_test):
    lr = LogisticRegression()
    lr.fit(y_prob_train.reshape(-1, 1), y_true_train)
    return lr.predict_proba(y_prob_test.reshape(-1, 1))[:, 1]

print("✓ Fonctions de recalibration prêtes")

In [ ]:
# 10.2 — Recalibration post-hoc : split 70/30 train/test
rng = np.random.default_rng(42)
resultats_recal = []

for fichier in sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet")):
    nom = os.path.basename(fichier).replace('_predictions.parquet', '')
    dfp = pd.read_parquet(fichier).copy()
    dfp['correct'] = (dfp['pred'] == dfp['expert']).astype(int)

    n = len(dfp)
    idx = rng.permutation(n)
    n_train = int(0.7 * n)
    train_idx, test_idx = idx[:n_train], idx[n_train:]

    y_true_train = dfp['correct'].iloc[train_idx].values
    y_prob_train = dfp['confidence'].iloc[train_idx].values
    y_true_test = dfp['correct'].iloc[test_idx].values
    y_prob_test = dfp['confidence'].iloc[test_idx].values

    # ECE original (test set)
    ece_orig, _ = expected_calibration_error(y_true_test, y_prob_test)

    # Temperature scaling
    T = temperature_scaling_fit(y_true_train, y_prob_train)
    y_prob_ts = apply_temperature(y_prob_test, T)
    ece_ts, _ = expected_calibration_error(y_true_test, y_prob_ts)

    # Isotonic
    y_prob_iso = isotonic_calibrate(y_true_train, y_prob_train, y_prob_test)
    ece_iso, _ = expected_calibration_error(y_true_test, y_prob_iso)

    # Platt
    y_prob_platt = platt_calibrate(y_true_train, y_prob_train, y_prob_test)
    ece_platt, _ = expected_calibration_error(y_true_test, y_prob_platt)

    resultats_recal.append({
        'model': nom,
        'ece_original': float(ece_orig),
        'ece_temperature': float(ece_ts),
        'ece_isotonic': float(ece_iso),
        'ece_platt': float(ece_platt),
        'temperature': float(T),
        'gain_temperature': float(ece_orig - ece_ts),
        'gain_isotonic': float(ece_orig - ece_iso),
        'gain_platt': float(ece_orig - ece_platt),
    })
    print(f"{nom:25} | orig={ece_orig:.3f} | TS={ece_ts:.3f} (T={T:.2f}) | iso={ece_iso:.3f} | Platt={ece_platt:.3f}")

recal_df = pd.DataFrame(resultats_recal)
recal_df.to_csv(f"{OUTPUT}/recalibration_post_hoc.csv", index=False)

PARTIE 11 — Prédiction conforme


In [ ]:
# 11 — Prédiction conforme : garantie de couverture statistique
# Pour chaque cas, on retourne un ENSEMBLE de labels possibles tel que la vraie réponse
# y soit avec probabilité ≥ 1-α (ex: 90%)

def conformal_predict_sets(df_pred, alpha=0.1):
    """
    Prédiction conforme split-conformal.
    Retourne les tailles d'ensembles et la couverture empirique.
    """
    dfp = df_pred.copy()
    n = len(dfp)
    idx = np.random.default_rng(42).permutation(n)
    n_cal = int(0.5 * n)
    cal_idx, test_idx = idx[:n_cal], idx[n_cal:]

    # Pour chaque calibration : score de non-conformité = 1 - p(vrai label)
    cols_proba = [c for c in dfp.columns if c.startswith('p_')]

    def get_proba_true(row):
        true_lab = row['expert'].replace(' ', '_')
        col = f'p_{true_lab}'
        return row[col] if col in row.index else np.nan

    dfp['proba_true'] = dfp.apply(get_proba_true, axis=1)
    dfp['non_conform'] = 1 - dfp['proba_true']

    # Quantile sur la calibration
    nonconf_cal = dfp['non_conform'].iloc[cal_idx].dropna().values
    q = np.quantile(nonconf_cal, 1 - alpha)

    # Sur le test set : pour chaque exemple, retenir tous les labels avec (1 - p) <= q
    couverture_empirique = 0
    tailles = []
    for _, row in dfp.iloc[test_idx].iterrows():
        labels_inclus = []
        for col in cols_proba:
            lab = col.replace('p_', '').replace('_', ' ')
            p = row[col]
            if pd.notna(p) and (1 - p) <= q:
                labels_inclus.append(lab)
        tailles.append(len(labels_inclus))
        if row['expert'] in labels_inclus:
            couverture_empirique += 1

    return {
        'couverture_visee': 1 - alpha,
        'couverture_empirique': couverture_empirique / len(test_idx),
        'taille_moyenne': float(np.mean(tailles)),
        'taille_mediane': float(np.median(tailles)),
        'quantile': float(q)
    }

print("=== Prédiction conforme (α=0.10, couverture visée 90%) ===\n")
conformal_res = []
for fichier in sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet")):
    nom = os.path.basename(fichier).replace('_predictions.parquet', '')
    dfp = pd.read_parquet(fichier)
    res = conformal_predict_sets(dfp, alpha=0.1)
    res['model'] = nom
    conformal_res.append(res)
    print(f"{nom:25} | couv. empir.={res['couverture_empirique']:.3f} | "
          f"taille moy.={res['taille_moyenne']:.2f}")

conformal_df = pd.DataFrame(conformal_res)
conformal_df.to_csv(f"{OUTPUT}/predictions_conformes.csv", index=False)

PARTIE 12 — Reliability diagrams par label


In [ ]:
# 12 — Calibration par label de sortie
def ece_par_label(df_pred, model_name):
    """ECE séparé pour chaque label prédit."""
    dfp = df_pred.copy()
    dfp['correct'] = (dfp['pred'] == dfp['expert']).astype(int)
    resultats = []
    for lab in dfp['pred'].unique():
        sub = dfp[dfp['pred'] == lab]
        if len(sub) < 30:
            continue
        ece, _ = expected_calibration_error(
            sub['correct'].values, sub['confidence'].values, n_bins=10
        )
        resultats.append({
            'model': model_name, 'label': lab,
            'n': len(sub),
            'accuracy': float(sub['correct'].mean()),
            'mean_confidence': float(sub['confidence'].mean()),
            'ece': float(ece)
        })
    return resultats

ece_label = []
for fichier in sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet")):
    nom = os.path.basename(fichier).replace('_predictions.parquet', '')
    dfp = pd.read_parquet(fichier)
    ece_label.extend(ece_par_label(dfp, nom))

ece_label_df = pd.DataFrame(ece_label)
print("ECE par label de sortie :\n")
print(ece_label_df.to_string(index=False))
ece_label_df.to_csv(f"{OUTPUT}/ece_par_label.csv", index=False)

PARTIE 13 — Effet de la longueur du critère


In [ ]:
# 13 — La calibration se dégrade-t-elle pour les critères longs ?
df['criterion_length'] = df['criterion_text'].str.len()
quartiles = df['criterion_length'].quantile([0.25, 0.5, 0.75]).values
print(f"Quartiles longueur critère : Q1={quartiles[0]:.0f}, Q2={quartiles[1]:.0f}, Q3={quartiles[2]:.0f}")

resultats_long = []
for fichier in sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet")):
    nom = os.path.basename(fichier).replace('_predictions.parquet', '')
    dfp = pd.read_parquet(fichier).copy()
    dfp = dfp.merge(df[['annotation_id', 'criterion_length']], on='annotation_id', how='left')
    dfp['correct'] = (dfp['pred'] == dfp['expert']).astype(int)

    for label_quartile, mask in [
        ('Q1 (court)', dfp['criterion_length'] <= quartiles[0]),
        ('Q2', (dfp['criterion_length'] > quartiles[0]) & (dfp['criterion_length'] <= quartiles[1])),
        ('Q3', (dfp['criterion_length'] > quartiles[1]) & (dfp['criterion_length'] <= quartiles[2])),
        ('Q4 (long)', dfp['criterion_length'] > quartiles[2]),
    ]:
        sub = dfp[mask]
        if len(sub) < 30:
            continue
        ece, _ = expected_calibration_error(sub['correct'].values, sub['confidence'].values)
        resultats_long.append({
            'model': nom,
            'quartile': label_quartile,
            'n': len(sub),
            'accuracy': float(sub['correct'].mean()),
            'ece': float(ece)
        })

long_df = pd.DataFrame(resultats_long)
print("\nECE par quartile de longueur du critère :\n")
print(long_df.to_string(index=False))
long_df.to_csv(f"{OUTPUT}/ece_par_longueur.csv", index=False)

PARTIE 14 — Cohérence inter-modèles

In [ ]:
# 14 — Quand les modèles s'accordent, sont-ils mieux calibrés ?
# On prend les 6 modèles en prompt B et on compte combien sont d'accord par paire

modeles_B = ['mistral_instruct_B', 'mistral_base_B', 'llama_instruct_B',
             'llama_base_B', 'meditron_B', 'biomistral_B']

dfs = {}
for nom in modeles_B:
    f = f"{OUTPUT}/{nom}_predictions.parquet"
    if os.path.exists(f):
        dfs[nom] = pd.read_parquet(f).set_index('annotation_id')

# Construire un DataFrame avec les prédictions de chaque modèle
preds_matrix = pd.DataFrame({nom: dfs[nom]['pred'] for nom in dfs})
preds_matrix['expert'] = dfs[modeles_B[0]]['expert']

# Compter le nombre de modèles d'accord avec l'expert pour chaque paire
preds_matrix['n_accord_expert'] = sum((preds_matrix[m] == preds_matrix['expert']).astype(int)
                                       for m in dfs)

print("Distribution du nombre de modèles d'accord avec l'expert :")
print(preds_matrix['n_accord_expert'].value_counts().sort_index())

# Pour chaque niveau de consensus, calculer l'accuracy moyenne
print("\nAccuracy moyenne selon le consensus :")
for n_acc in range(7):
    sub = preds_matrix[preds_matrix['n_accord_expert'] == n_acc]
    if len(sub) > 0:
        print(f"  {n_acc}/6 modèles d'accord avec expert : {len(sub)} cas")

# Quand TOUS les modèles s'accordent entre eux, sont-ils mieux ?
preds_matrix['tous_accord'] = preds_matrix[list(dfs)].nunique(axis=1) == 1
accord_quand_unanime = (preds_matrix[preds_matrix['tous_accord']][list(dfs)[0]] ==
                       preds_matrix[preds_matrix['tous_accord']]['expert']).mean()
print(f"\nQuand les 6 modèles s'accordent unanimement (n={preds_matrix['tous_accord'].sum()}) :")
print(f"  Accuracy de leur consensus : {accord_quand_unanime:.3f}")

In [ ]:
# ============================================================================
# PARTIE 15 — Vérification de robustesse de la recalibration isotonic
# ============================================================================
# CELLULE AUTONOME : réimporte tout ce qui est nécessaire pour une session
# fraîche où les .parquet sont sauvés dans /kaggle/working/results/
# ============================================================================

import os
import glob
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import KFold

# --- Chemins ---
OUTPUT = "/kaggle/working/results"


# --- Fonctions de calibration (réimportées) ---
def expected_calibration_error(y_true, y_prob, n_bins=10):
    """ECE avec binning uniforme."""
    bins = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    ece = 0.0
    bin_data = []
    for k in range(n_bins):
        mask = (bin_ids == k)
        if mask.sum() > 0:
            acc_k = y_true[mask].mean()
            conf_k = y_prob[mask].mean()
            weight = mask.sum() / len(y_true)
            ece += weight * abs(acc_k - conf_k)
            bin_data.append({
                'bin': k, 'low': bins[k], 'high': bins[k+1],
                'count': int(mask.sum()),
                'accuracy': float(acc_k),
                'confidence': float(conf_k),
                'gap': float(abs(acc_k - conf_k)),
            })
    return ece, pd.DataFrame(bin_data)


def isotonic_calibrate(y_true_train, y_prob_train, y_prob_test):
    """Recalibration isotonic monotone."""
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(y_prob_train, y_true_train)
    return iso.predict(y_prob_test)


# --- Fonction de vérification (métrique reformulée) ---
def verifier_recalibration(df_pred, model_name, n_splits=5):
    """
    Vérifie la robustesse de la recalibration isotonic par cross-validation.

    Métriques :
    - ece_orig, ece_iso : ECE avant et après recalibration
    - gain_ece : ECE_orig - ECE_iso (positif = amélioration)
    - confidence_spread_ratio : std(iso) / std(orig)
        * ratio < 1 : les confiances sont écrasées (perte de discrimination)
        * ratio = 1 : discrimination préservée
        * ratio > 1 : les confiances sont étalées (gain de discrimination)
    """
    dfp = df_pred.copy()
    dfp['correct'] = (dfp['pred'] == dfp['expert']).astype(int)
    y_true = dfp['correct'].values
    y_prob = dfp['confidence'].values

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    resultats = []

    for i, (train_idx, test_idx) in enumerate(kf.split(y_true)):
        ece_orig, _ = expected_calibration_error(y_true[test_idx], y_prob[test_idx])
        y_prob_iso = isotonic_calibrate(y_true[train_idx], y_prob[train_idx], y_prob[test_idx])
        ece_iso, _ = expected_calibration_error(y_true[test_idx], y_prob_iso)

        std_orig = float(y_prob[test_idx].std())
        std_iso = float(y_prob_iso.std())
        confidence_spread_ratio = std_iso / std_orig if std_orig > 0 else 1.0

        accuracy = float(y_true[test_idx].mean())
        below_threshold = ece_iso < 0.05

        resultats.append({
            'model': model_name,
            'fold': i,
            'ece_orig': float(ece_orig),
            'ece_iso': float(ece_iso),
            'gain_ece': float(ece_orig - ece_iso),
            'accuracy': accuracy,
            'std_conf_orig': std_orig,
            'std_conf_iso': std_iso,
            'confidence_spread_ratio': float(confidence_spread_ratio),
            'below_threshold': below_threshold,
        })

    return pd.DataFrame(resultats)


# ============================================================================
# LANCEMENT
# ============================================================================
print("=" * 80)
print(" VÉRIFICATION DE ROBUSTESSE — Recalibration isotonic (5-fold CV)")
print("=" * 80)

fichiers = sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet"))
if len(fichiers) == 0:
    print(f"\n⚠ Aucun fichier trouvé dans {OUTPUT}")
else:
    print(f"\n✓ {len(fichiers)} fichiers trouvés\n")

    tous_resultats_verif = []
    for fichier in fichiers:
        nom = os.path.basename(fichier).replace('_predictions.parquet', '')
        dfp = pd.read_parquet(fichier)
        res_verif = verifier_recalibration(dfp, nom, n_splits=5)
        tous_resultats_verif.append(res_verif)

    verif_df = pd.concat(tous_resultats_verif, ignore_index=True)

    # Résumé par modèle
    resume = verif_df.groupby('model').agg(
        ece_orig_mean=('ece_orig', 'mean'),
        ece_orig_std=('ece_orig', 'std'),
        ece_iso_mean=('ece_iso', 'mean'),
        ece_iso_std=('ece_iso', 'std'),
        gain_ece_mean=('gain_ece', 'mean'),
        confidence_spread_ratio_mean=('confidence_spread_ratio', 'mean'),
        prop_below_threshold=('below_threshold', 'mean'),
    ).round(4)

    # Interprétation textuelle de spread_ratio
    def interpreter_spread(ratio):
        if ratio > 1.15:
            return "étale (+)"
        elif ratio < 0.85:
            return "écrase (-)"
        else:
            return "stable"

    resume['comportement'] = resume['confidence_spread_ratio_mean'].apply(interpreter_spread)

    print("=" * 90)
    print(" RÉSUMÉ PAR MODÈLE (moyennes sur 5 folds)")
    print("=" * 90 + "\n")
    print(resume.to_string())

    print("\n" + "-" * 90)
    print(" INTERPRÉTATION")
    print("-" * 90)
    print("- ece_iso_mean < 0.05 : modèle sous le seuil de Van Calster après recalibration")
    print("- prop_below_threshold : proportion des 5 folds passant sous le seuil")
    print("- confidence_spread_ratio :")
    print("    > 1.15 → l'isotonic étale les confiances (gain de discrimination)")
    print("    0.85-1.15 → discrimination préservée")
    print("    < 0.85 → l'isotonic écrase les confiances (perte de discrimination)")

    os.makedirs(OUTPUT, exist_ok=True)
    resume.to_csv(f"{OUTPUT}/verification_robustesse_recalibration.csv")
    verif_df.to_csv(f"{OUTPUT}/verification_robustesse_recalibration_detail.csv", index=False)

    print(f"\n✓ Résumé sauvegardé : {OUTPUT}/verification_robustesse_recalibration.csv")

In [ ]:
# ============================================================================
# PARTIE 16 — Comparaison avec baselines naïves
# ============================================================================
# Trois baselines :
#   1. Majorité : prédit toujours le label le plus fréquent chez les experts
#   2. Uniforme : prédit chaque label avec proba 1/K, confiance = 1/K
#   3. NEI-baseline : prédit toujours "not enough information", confiance 0.5
# ============================================================================

import os
import glob
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss

OUTPUT = "/kaggle/working/results"

# Charger un fichier de référence pour la distribution experte
df_ref = pd.read_parquet(f"{OUTPUT}/mistral_instruct_B_predictions.parquet")

# Statistiques expertes
label_freq = df_ref['expert'].value_counts(normalize=True)
label_majoritaire = label_freq.index[0]
print(f"Label le plus fréquent chez les experts : '{label_majoritaire}' ({label_freq.iloc[0]:.1%})")


def ece_baseline(confidences, corrections):
    """Petit wrapper ECE pour baselines."""
    bins = np.linspace(0, 1, 11)
    bin_ids = np.clip(np.digitize(confidences, bins) - 1, 0, 9)
    ece = 0.0
    for k in range(10):
        mask = (bin_ids == k)
        if mask.sum() > 0:
            gap = abs(corrections[mask].mean() - confidences[mask].mean())
            ece += (mask.sum() / len(confidences)) * gap
    return ece


baselines = []

# Baseline 1 — Majorité (prédit toujours le label majoritaire)
pred_maj = [label_majoritaire] * len(df_ref)
correct_maj = np.array([1 if p == e else 0 for p, e in zip(pred_maj, df_ref['expert'])])
conf_maj = np.full(len(df_ref), label_freq.iloc[0])
baselines.append({
    'baseline': f"Majority ('{label_majoritaire}')",
    'accuracy': correct_maj.mean(),
    'ece': ece_baseline(conf_maj, correct_maj),
    'brier': brier_score_loss(correct_maj, conf_maj),
    'mean_confidence': conf_maj.mean(),
})

# Baseline 2 — Uniforme (predict aléatoirement parmi 6 labels avec conf 1/6)
rng = np.random.default_rng(42)
pred_unif = rng.choice(list(label_freq.index), size=len(df_ref))
correct_unif = np.array([1 if p == e else 0 for p, e in zip(pred_unif, df_ref['expert'])])
conf_unif = np.full(len(df_ref), 1/6)
baselines.append({
    'baseline': 'Uniform (random pick, conf=1/6)',
    'accuracy': correct_unif.mean(),
    'ece': ece_baseline(conf_unif, correct_unif),
    'brier': brier_score_loss(correct_unif, conf_unif),
    'mean_confidence': conf_unif.mean(),
})

# Baseline 3 — Toujours "not enough information" avec confiance 0.5
pred_nei = ['not enough information'] * len(df_ref)
correct_nei = np.array([1 if p == e else 0 for p, e in zip(pred_nei, df_ref['expert'])])
conf_nei = np.full(len(df_ref), 0.5)
baselines.append({
    'baseline': "Always 'not enough info', conf=0.5",
    'accuracy': correct_nei.mean(),
    'ece': ece_baseline(conf_nei, correct_nei),
    'brier': brier_score_loss(correct_nei, conf_nei),
    'mean_confidence': conf_nei.mean(),
})

baselines_df = pd.DataFrame(baselines).round(4)
print("\n=== BASELINES NAÏVES ===\n")
print(baselines_df.to_string(index=False))

# Comparaison avec le meilleur LLM (Meditron A)
med_a = pd.read_parquet(f"{OUTPUT}/meditron_A_predictions.parquet")
med_correct = (med_a['pred'] == med_a['expert']).astype(int).values
print("\n=== BEST LLM (Meditron-A, brut) ===")
print(f"Accuracy : {med_correct.mean():.4f}")
print(f"ECE      : {ece_baseline(med_a['confidence'].values, med_correct):.4f}")

baselines_df.to_csv(f"{OUTPUT}/baselines_naives.csv", index=False)
print(f"\n✓ Sauvegardé : {OUTPUT}/baselines_naives.csv")

In [ ]:
# ============================================================================
# PARTIE 16b — Approfondissement de la comparaison avec baselines
# ============================================================================

import os
import glob
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix
from scipy.stats import entropy

OUTPUT = "/kaggle/working/results"

# Charger un fichier de référence pour la vérité experte
df_ref = pd.read_parquet(f"{OUTPUT}/mistral_instruct_B_predictions.parquet")
expert_labels = df_ref['expert'].values
labels_uniques = sorted(df_ref['expert'].unique().tolist())
label_majoritaire = df_ref['expert'].value_counts().idxmax()

print(f"Distribution experte :")
print(df_ref['expert'].value_counts())
print(f"\nLabel majoritaire : '{label_majoritaire}'")


# ============================================================================
# ANALYSE 1 — F1 macro pour tous les modèles + baselines
# ============================================================================
print("\n" + "="*80)
print(" ANALYSE 1 — F1 macro (pénalise les modèles qui ignorent les classes)")
print("="*80)

def ece_simple(y_true_bin, y_prob):
    """ECE binaire simple."""
    bins = np.linspace(0, 1, 11)
    bin_ids = np.clip(np.digitize(y_prob, bins) - 1, 0, 9)
    ece = 0.0
    for k in range(10):
        mask = (bin_ids == k)
        if mask.sum() > 0:
            gap = abs(y_true_bin[mask].mean() - y_prob[mask].mean())
            ece += (mask.sum() / len(y_prob)) * gap
    return ece

resultats_f1 = []

# Baseline Majority
pred_maj = np.array([label_majoritaire] * len(expert_labels))
resultats_f1.append({
    'model': 'BASELINE_Majority',
    'accuracy': (pred_maj == expert_labels).mean(),
    'f1_macro': f1_score(expert_labels, pred_maj, average='macro', zero_division=0),
    'f1_weighted': f1_score(expert_labels, pred_maj, average='weighted', zero_division=0),
    'nb_labels_predits': len(np.unique(pred_maj)),
})

# Baseline NEI
pred_nei = np.array(['not enough information'] * len(expert_labels))
resultats_f1.append({
    'model': 'BASELINE_AlwaysNEI',
    'accuracy': (pred_nei == expert_labels).mean(),
    'f1_macro': f1_score(expert_labels, pred_nei, average='macro', zero_division=0),
    'f1_weighted': f1_score(expert_labels, pred_nei, average='weighted', zero_division=0),
    'nb_labels_predits': len(np.unique(pred_nei)),
})

# Tous les LLM
for fichier in sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet")):
    nom = os.path.basename(fichier).replace('_predictions.parquet', '')
    dfp = pd.read_parquet(fichier)
    resultats_f1.append({
        'model': nom,
        'accuracy': (dfp['pred'] == dfp['expert']).mean(),
        'f1_macro': f1_score(dfp['expert'], dfp['pred'], average='macro', zero_division=0),
        'f1_weighted': f1_score(dfp['expert'], dfp['pred'], average='weighted', zero_division=0),
        'nb_labels_predits': dfp['pred'].nunique(),
    })

f1_df = pd.DataFrame(resultats_f1).round(4)
f1_df = f1_df.sort_values('f1_macro', ascending=False)
print(f1_df.to_string(index=False))


# ============================================================================
# ANALYSE 2 — Performance sur les cas NON triviaux
# ============================================================================
print("\n" + "="*80)
print(" ANALYSE 2 — Accuracy sur les cas 'difficiles' (vérité ≠ label majoritaire)")
print("="*80)

# Masque des cas non triviaux (vérité experte différente de 'not excluded')
mask_difficile = expert_labels != label_majoritaire
n_diff = mask_difficile.sum()
n_facile = (~mask_difficile).sum()
print(f"\nCas 'faciles' (majoritaires) : {n_facile} ({n_facile/len(expert_labels)*100:.1f}%)")
print(f"Cas 'difficiles' (minoritaires) : {n_diff} ({n_diff/len(expert_labels)*100:.1f}%)")

resultats_difficile = []

# Baselines sur cas difficiles
for nom_base, pred_arr in [('BASELINE_Majority', pred_maj), ('BASELINE_AlwaysNEI', pred_nei)]:
    acc_diff = (pred_arr[mask_difficile] == expert_labels[mask_difficile]).mean()
    resultats_difficile.append({
        'model': nom_base,
        'accuracy_difficile': acc_diff,
    })

# LLMs sur cas difficiles
for fichier in sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet")):
    nom = os.path.basename(fichier).replace('_predictions.parquet', '')
    dfp = pd.read_parquet(fichier)
    preds = dfp['pred'].values
    acc_diff = (preds[mask_difficile] == expert_labels[mask_difficile]).mean()
    resultats_difficile.append({
        'model': nom,
        'accuracy_difficile': acc_diff,
    })

diff_df = pd.DataFrame(resultats_difficile).round(4)
diff_df = diff_df.sort_values('accuracy_difficile', ascending=False)
print(diff_df.to_string(index=False))


# ============================================================================
# ANALYSE 3 — Est-ce que les modèles imitent les baselines ?
# ============================================================================
print("\n" + "="*80)
print(" ANALYSE 3 — Distribution des prédictions (% par label)")
print("="*80)

distribution_predictions = {}

# Experts
dist_expert = pd.Series(expert_labels).value_counts(normalize=True).round(3)
distribution_predictions['expert'] = dist_expert

for fichier in sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet")):
    nom = os.path.basename(fichier).replace('_predictions.parquet', '')
    dfp = pd.read_parquet(fichier)
    dist = dfp['pred'].value_counts(normalize=True).round(3)
    distribution_predictions[nom] = dist

dist_df = pd.DataFrame(distribution_predictions).fillna(0).T
print("Proportion de chaque label prédit par modèle :\n")
print(dist_df.to_string())

# Détection : est-ce que certains modèles s'effondrent sur un label ?
print("\n--- Modèles suspects (>60% sur un seul label) ---")
for model_name, row in dist_df.iterrows():
    max_pct = row.max()
    max_label = row.idxmax()
    if max_pct > 0.60 and model_name != 'expert':
        print(f"  {model_name:25} : {max_pct*100:.1f}% de '{max_label}'")


# ============================================================================
# ANALYSE 4 — Score composite d'utilité clinique
# ============================================================================
print("\n" + "="*80)
print(" ANALYSE 4 — Score composite d'utilité clinique")
print("="*80)

# Score composite = (F1 macro) * (1 - ECE) * (diversité des prédictions)
# Diversité = entropie normalisée des prédictions (0 = un seul label, 1 = uniforme)

def diversite_normalisee(preds, n_labels=6):
    """Entropie de Shannon normalisée [0, 1]."""
    freq = pd.Series(preds).value_counts(normalize=True).values
    ent = entropy(freq, base=2)
    return ent / np.log2(n_labels)

# Charger les ECE originaux (déjà calculés)
try:
    recap_calib = pd.read_csv(f"{OUTPUT}/recap_calibration_tous_modeles.csv")
    ece_map = dict(zip(recap_calib['model'], recap_calib['ece']))
except:
    ece_map = {}

composite = []
for _, row in f1_df.iterrows():
    nom = row['model']
    if nom.startswith('BASELINE'):
        continue
    dfp = pd.read_parquet(f"{OUTPUT}/{nom}_predictions.parquet")
    div = diversite_normalisee(dfp['pred'].values)
    ece = ece_map.get(nom, np.nan)
    if not np.isnan(ece):
        # Score composite
        score = row['f1_macro'] * (1 - min(ece, 1.0)) * div
        composite.append({
            'model': nom,
            'f1_macro': row['f1_macro'],
            'ece': ece,
            'calibration_quality': 1 - min(ece, 1.0),
            'diversity': div,
            'composite_score': score,
        })

comp_df = pd.DataFrame(composite).round(4)
comp_df = comp_df.sort_values('composite_score', ascending=False)
print("\nScore composite (F1_macro × (1-ECE) × diversité) :\n")
print(comp_df.to_string(index=False))


# ============================================================================
# SAUVEGARDE
# ============================================================================
f1_df.to_csv(f"{OUTPUT}/analyse_f1_macro.csv", index=False)
diff_df.to_csv(f"{OUTPUT}/analyse_cas_difficiles.csv", index=False)
dist_df.to_csv(f"{OUTPUT}/analyse_distribution_predictions.csv")
comp_df.to_csv(f"{OUTPUT}/score_composite_utilite.csv", index=False)

print("\n" + "="*80)
print(f"✓ Résultats sauvegardés dans {OUTPUT}/")
print("="*80)

In [ ]:
# ============================================================================
# PARTIE 18 — Analyse contre-factuelle : performance sur les cas non-triviaux
# ============================================================================
# On retire les paires où la vérité experte est "not excluded" ou 
# "not enough information" (les deux labels dominants qui masquent
# la capacité réelle de discrimination des modèles).
#
# Question : sur les cas qui exigent une vraie décision clinique, que font
#            les modèles ?
# ============================================================================

import os
import glob
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix

OUTPUT = "/kaggle/working/results"

LABELS_DOMINANTS = ['not excluded', 'not enough information']


def expected_calibration_error(y_true, y_prob, n_bins=10):
    """ECE binaire."""
    bins = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.clip(np.digitize(y_prob, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for k in range(n_bins):
        mask = (bin_ids == k)
        if mask.sum() > 0:
            gap = abs(y_true[mask].mean() - y_prob[mask].mean())
            ece += (mask.sum() / len(y_true)) * gap
    return ece


print("=" * 90)
print(" ANALYSE CONTRE-FACTUELLE — cas non-triviaux uniquement")
print(" (retrait des paires où l'expert a annoté 'not excluded' ou 'NEI')")
print("=" * 90)

# Statistiques préalables
df_ref = pd.read_parquet(f"{OUTPUT}/mistral_instruct_B_predictions.parquet")
n_total = len(df_ref)
mask_non_trivial = ~df_ref['expert'].isin(LABELS_DOMINANTS)
n_non_trivial = mask_non_trivial.sum()

print(f"\nTotal des paires    : {n_total}")
print(f"Cas dominants       : {n_total - n_non_trivial} ({(n_total - n_non_trivial)/n_total*100:.1f}%)")
print(f"Cas non-triviaux    : {n_non_trivial} ({n_non_trivial/n_total*100:.1f}%)")

print(f"\nDistribution des cas non-triviaux (vérité experte) :")
print(df_ref[mask_non_trivial]['expert'].value_counts())


# ============================================================================
# ANALYSE COMPARÉE
# ============================================================================
resultats = []

for fichier in sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet")):
    nom = os.path.basename(fichier).replace('_predictions.parquet', '')
    dfp = pd.read_parquet(fichier)

    # Sous-ensemble non-trivial
    dfp_nt = dfp[~dfp['expert'].isin(LABELS_DOMINANTS)].copy()
    dfp_nt['correct'] = (dfp_nt['pred'] == dfp_nt['expert']).astype(int)

    if len(dfp_nt) < 30:
        continue

    # Métriques sur les cas non-triviaux
    accuracy_nt = dfp_nt['correct'].mean()
    f1_macro_nt = f1_score(dfp_nt['expert'], dfp_nt['pred'],
                            average='macro', zero_division=0)
    ece_nt = expected_calibration_error(dfp_nt['correct'].values,
                                         dfp_nt['confidence'].values)

    # Combien de fois le modèle prédit un label non-trivial ?
    predit_non_trivial = (~dfp_nt['pred'].isin(LABELS_DOMINANTS)).sum()
    prop_predit_non_trivial = predit_non_trivial / len(dfp_nt)

    # Précision quand le modèle ose une prédiction non-triviale
    dfp_nt_ose = dfp_nt[~dfp_nt['pred'].isin(LABELS_DOMINANTS)]
    accuracy_quand_ose = dfp_nt_ose['correct'].mean() if len(dfp_nt_ose) > 0 else 0.0

    resultats.append({
        'model': nom,
        'n_cas_non_triviaux': len(dfp_nt),
        'accuracy_non_trivial': accuracy_nt,
        'f1_macro_non_trivial': f1_macro_nt,
        'ece_non_trivial': ece_nt,
        'prop_predit_non_trivial': prop_predit_non_trivial,
        'accuracy_quand_ose_predire': accuracy_quand_ose,
        'n_fois_ose': predit_non_trivial,
    })


res_df = pd.DataFrame(resultats).round(4)
res_df = res_df.sort_values('f1_macro_non_trivial', ascending=False)

print("\n" + "=" * 90)
print(" PERFORMANCE SUR LES CAS NON-TRIVIAUX")
print("=" * 90 + "\n")
print(res_df[['model', 'accuracy_non_trivial', 'f1_macro_non_trivial',
              'ece_non_trivial', 'prop_predit_non_trivial',
              'accuracy_quand_ose_predire']].to_string(index=False))


# ============================================================================
# COMPARAISON : ACCURACY GLOBALE vs NON-TRIVIAL
# ============================================================================
print("\n" + "=" * 90)
print(" COMPARAISON : cas dominants vs cas non-triviaux (chute de performance)")
print("=" * 90 + "\n")

comparaison = []
for fichier in sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet")):
    nom = os.path.basename(fichier).replace('_predictions.parquet', '')
    dfp = pd.read_parquet(fichier)

    dfp['correct'] = (dfp['pred'] == dfp['expert']).astype(int)
    acc_global = dfp['correct'].mean()

    dfp_dom = dfp[dfp['expert'].isin(LABELS_DOMINANTS)]
    acc_dominant = dfp_dom['correct'].mean()

    dfp_nt = dfp[~dfp['expert'].isin(LABELS_DOMINANTS)]
    acc_non_trivial = dfp_nt['correct'].mean()

    comparaison.append({
        'model': nom,
        'accuracy_globale': acc_global,
        'accuracy_cas_dominants': acc_dominant,
        'accuracy_cas_non_triviaux': acc_non_trivial,
        'chute_performance': acc_dominant - acc_non_trivial,
    })

comp_df = pd.DataFrame(comparaison).round(4)
comp_df = comp_df.sort_values('accuracy_cas_non_triviaux', ascending=False)
print(comp_df.to_string(index=False))


# ============================================================================
# SAUVEGARDE
# ============================================================================
res_df.to_csv(f"{OUTPUT}/analyse_cas_non_triviaux.csv", index=False)
comp_df.to_csv(f"{OUTPUT}/comparaison_dominants_vs_non_triviaux.csv", index=False)
print(f"\n✓ Sauvegardé : {OUTPUT}/analyse_cas_non_triviaux.csv")
print(f"✓ Sauvegardé : {OUTPUT}/comparaison_dominants_vs_non_triviaux.csv")

In [ ]:
# ============================================================================
# PARTIE 19 — Simulation : modèles forcés à trancher (renormalisation)
# ============================================================================
# À partir des distributions déjà stockées, on renormalise sur seulement 2 
# labels décisifs par type de critère :
#   - Inclusion : {included, not_included}
#   - Exclusion : {excluded, not_excluded}
# Les labels d'esquive (not_enough_information, not_applicable) sont retirés.
# On évalue uniquement sur les cas où l'expert lui-même a tranché.
# ============================================================================

import os
import glob
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

OUTPUT = "/kaggle/working/results"


def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.clip(np.digitize(y_prob, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for k in range(n_bins):
        mask = (bin_ids == k)
        if mask.sum() > 0:
            gap = abs(y_true[mask].mean() - y_prob[mask].mean())
            ece += (mask.sum() / len(y_true)) * gap
    return ece


def forcer_trancher(dfp):
    """
    Recalcule les prédictions en forçant le choix entre labels décisifs.
    """
    dfp = dfp.copy()
    new_preds = []
    new_confidences = []
    p_pos_list = []  # proba de la classe "positive" pour ECE binaire

    for _, row in dfp.iterrows():
        if row['criterion_type'] == 'inclusion':
            candidats = ['included', 'not included']
            probs = np.array([
                row['p_included'] if not pd.isna(row['p_included']) else 0,
                row['p_not_included'] if not pd.isna(row['p_not_included']) else 0,
            ])
        else:
            candidats = ['excluded', 'not excluded']
            probs = np.array([
                row['p_excluded'] if not pd.isna(row['p_excluded']) else 0,
                row['p_not_excluded'] if not pd.isna(row['p_not_excluded']) else 0,
            ])

        # Renormalisation
        if probs.sum() > 0:
            probs = probs / probs.sum()
        else:
            probs = np.array([0.5, 0.5])

        idx_max = int(np.argmax(probs))
        new_preds.append(candidats[idx_max])
        new_confidences.append(float(probs[idx_max]))
        p_pos_list.append(float(probs[0]))  # proba du premier candidat

    dfp['new_pred'] = new_preds
    dfp['new_confidence'] = new_confidences
    dfp['p_pos'] = p_pos_list
    return dfp


# ============================================================================
# LANCEMENT
# ============================================================================
print("=" * 95)
print(" SIMULATION : modèles FORCÉS à trancher (2 labels décisifs par type)")
print("=" * 95)

resultats_forces = []

for fichier in sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet")):
    nom = os.path.basename(fichier).replace('_predictions.parquet', '')
    dfp = pd.read_parquet(fichier)

    # Ne comparer que sur les cas où l'expert a effectivement tranché
    mask_expert_tranche = ~dfp['expert'].isin(['not enough information', 'not applicable'])
    dfp_test = dfp[mask_expert_tranche].copy()

    if len(dfp_test) < 30:
        continue

    dfp_force = forcer_trancher(dfp_test)
    dfp_force['correct'] = (dfp_force['new_pred'] == dfp_force['expert']).astype(int)

    accuracy = dfp_force['correct'].mean()
    ece = expected_calibration_error(
        dfp_force['correct'].values, dfp_force['new_confidence'].values
    )
    f1 = f1_score(dfp_force['expert'], dfp_force['new_pred'],
                   average='macro', zero_division=0)

    # Diversité des nouvelles prédictions
    dist = dfp_force['new_pred'].value_counts(normalize=True)
    max_prop = float(dist.iloc[0]) if len(dist) > 0 else 1.0

    # Statistiques par type de critère
    incl_mask = dfp_force['criterion_type'] == 'inclusion'
    excl_mask = dfp_force['criterion_type'] == 'exclusion'

    acc_incl = float(dfp_force[incl_mask]['correct'].mean()) if incl_mask.sum() > 0 else np.nan
    acc_excl = float(dfp_force[excl_mask]['correct'].mean()) if excl_mask.sum() > 0 else np.nan

    resultats_forces.append({
        'model': nom,
        'n_cas_tranche': len(dfp_test),
        'accuracy_force': accuracy,
        'ece_force': ece,
        'f1_macro_force': f1,
        'max_prop_label': max_prop,
        'acc_inclusion': acc_incl,
        'acc_exclusion': acc_excl,
    })


forces_df = pd.DataFrame(resultats_forces).round(4)
forces_df = forces_df.sort_values('f1_macro_force', ascending=False)

print("\n--- Performance quand les modèles sont FORCÉS à trancher ---\n")
print(forces_df.to_string(index=False))


# ============================================================================
# COMPARAISON AVANT / APRÈS
# ============================================================================
print("\n" + "=" * 95)
print(" COMPARAISON : distribution des prédictions avant/après contrainte")
print("=" * 95)

comparaison_dist = []
for fichier in sorted(glob.glob(f"{OUTPUT}/*_predictions.parquet")):
    nom = os.path.basename(fichier).replace('_predictions.parquet', '')
    dfp = pd.read_parquet(fichier)

    mask_expert_tranche = ~dfp['expert'].isin(['not enough information', 'not applicable'])
    dfp_test = dfp[mask_expert_tranche].copy()
    if len(dfp_test) < 30:
        continue

    dfp_force = forcer_trancher(dfp_test)

    # Distribution originale
    orig_dist = dfp_test['pred'].value_counts(normalize=True)
    orig_max = float(orig_dist.iloc[0])
    orig_max_label = orig_dist.index[0]

    # Distribution forcée
    forc_dist = dfp_force['new_pred'].value_counts(normalize=True)
    forc_max = float(forc_dist.iloc[0])
    forc_max_label = forc_dist.index[0]

    comparaison_dist.append({
        'model': nom,
        'orig_label_dominant': orig_max_label,
        'orig_%_dominant': f"{orig_max*100:.1f}%",
        'forcee_label_dominant': forc_max_label,
        'forcee_%_dominant': f"{forc_max*100:.1f}%",
        'sortie_evasion': orig_max_label in ['not enough information', 'not applicable'],
    })

comp_dist_df = pd.DataFrame(comparaison_dist)
print("\n")
print(comp_dist_df.to_string(index=False))


# ============================================================================
# INTERPRÉTATION
# ============================================================================
print("\n" + "=" * 95)
print(" INTERPRÉTATION")
print("=" * 95)
print()
print("Question 1 : les modèles dégénérés se 'réveillent'-ils quand forcés ?")
print("Question 2 : leur accuracy monte-t-elle au-dessus de 50% (mieux qu'aléatoire) ?")
print("Question 3 : leur ECE reste-t-il tolérable ?")
print()

for _, row in forces_df.iterrows():
    accuracy = row['accuracy_force']
    ece = row['ece_force']

    if accuracy > 0.65 and ece < 0.15:
        verdict = "🟢 Modèle avec vrai raisonnement clinique"
    elif accuracy > 0.55:
        verdict = "🟡 Modèle avec compétence limitée"
    else:
        verdict = "🔴 Modèle sans raisonnement clinique (aléatoire)"

    print(f"  {row['model']:25} | acc={accuracy:.3f} ECE={ece:.3f} → {verdict}")


forces_df.to_csv(f"{OUTPUT}/simulation_force_trancher.csv", index=False)
comp_dist_df.to_csv(f"{OUTPUT}/comparaison_distributions_forcees.csv", index=False)
print(f"\n✓ Sauvegardé : {OUTPUT}/simulation_force_trancher.csv")
print(f"✓ Sauvegardé : {OUTPUT}/comparaison_distributions_forcees.csv")

In [1]:
# ============================================================================
# PARTIE 20 — Weighted kappa expert vs GPT-4 avec IC 95% bootstrap
# ============================================================================
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

OUTPUT = "/kaggle/working/results"

# Charger n'importe quel fichier de prédiction (ils ont tous expert et gpt4)
df = pd.read_parquet(f"{OUTPUT}/mistral_instruct_B_predictions.parquet")

# Vérifier que la colonne gpt4 existe
print("Colonnes disponibles :", df.columns.tolist())
print(f"\nNombre de paires : {len(df)}")

# Si la colonne s'appelle 'gpt4', 'gpt_4', 'gpt4_label' ou autre, on adapte
# Regarde tes colonnes et remplace 'gpt4' par le vrai nom si nécessaire
COL_EXPERT = 'expert'
COL_GPT4 = 'gpt4'  # ← ADAPTE SI LE NOM EST DIFFÉRENT

# Vérification
if COL_GPT4 not in df.columns:
    print(f"\n⚠ Colonne '{COL_GPT4}' non trouvée. Colonnes disponibles :")
    for c in df.columns:
        print(f"  - {c}")
else:
    y_expert = df[COL_EXPERT].values
    y_gpt4 = df[COL_GPT4].values

    # Kappa quadratique point-estimate
    kappa_point = cohen_kappa_score(y_expert, y_gpt4, weights='quadratic')
    print(f"\n=== Weighted kappa (quadratic) — point estimate ===")
    print(f"κ_w = {kappa_point:.4f}")

    # Bootstrap CI 95%
    n_bootstrap = 1000
    n = len(df)
    rng = np.random.default_rng(42)
    kappas = []

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)  # tirage avec remise
        k = cohen_kappa_score(y_expert[idx], y_gpt4[idx], weights='quadratic')
        kappas.append(k)

    kappas = np.array(kappas)
    ci_low = np.percentile(kappas, 2.5)
    ci_high = np.percentile(kappas, 97.5)

    print(f"\n=== IC bootstrap 95% ({n_bootstrap} resamples) ===")
    print(f"κ_w = {kappa_point:.4f} [{ci_low:.4f}, {ci_high:.4f}]")

    # Résumé pour LaTeX
    print(f"\n=== Format LaTeX ===")
    print(f"$\\kappa_w = {kappa_point:.3f}$ (95\\% bootstrap CI: [{ci_low:.3f}, {ci_high:.3f}])")

    # Sauvegarder
    resultats_kappa = {
        'kappa_point': float(kappa_point),
        'ci_low_95': float(ci_low),
        'ci_high_95': float(ci_high),
        'n_bootstrap': n_bootstrap,
        'n_pairs': n,
    }
    pd.DataFrame([resultats_kappa]).to_csv(f"{OUTPUT}/kappa_expert_gpt4.csv", index=False)
    print(f"\n✓ Sauvegardé : {OUTPUT}/kappa_expert_gpt4.csv")

Colonnes disponibles : ['annotation_id', 'criterion_type', 'pred', 'confidence', 'expert', 'gpt4', 'p_included', 'p_not_included', 'p_excluded', 'p_not_excluded', 'p_not_enough_information', 'p_not_applicable']

Nombre de paires : 1015

=== Weighted kappa (quadratic) — point estimate ===
κ_w = 0.8188

=== IC bootstrap 95% (1000 resamples) ===
κ_w = 0.8188 [0.7745, 0.8600]

=== Format LaTeX ===
$\kappa_w = 0.819$ (95\% bootstrap CI: [0.774, 0.860])

✓ Sauvegardé : /kaggle/working/results/kappa_expert_gpt4.csv


In [ ]:
# ============================================================================
# PARTIE 21 — Reliability diagrams améliorés pour le mémoire
# ============================================================================

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

OUTPUT = "/kaggle/working/results"

# Style plus propre pour publication
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.linewidth': 0.8,
    'font.family': 'DejaVu Sans',
})

# Correspondance nom fichier → label lisible + couleur
MODELS_CONFIG = {
    'mistral_base':     {'label': 'Mistral-7B (Base)',        'color': '#E74C3C'},
    'mistral_instruct': {'label': 'Mistral-7B (Instruct)',    'color': '#F39C12'},
    'llama_base':       {'label': 'Llama-3.1-8B (Base)',      'color': '#3498DB'},
    'llama_instruct':   {'label': 'Llama-3.1-8B (Instruct)',  'color': '#2C3E50'},
    'meditron':         {'label': 'Meditron-7B',              'color': '#27AE60'},
    'biomistral':       {'label': 'BioMistral-7B',            'color': '#9B59B6'},
}


def compute_bins(y_true, y_prob, n_bins=10):
    """Calcule les bins pour le reliability diagram."""
    bins = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.clip(np.digitize(y_prob, bins) - 1, 0, n_bins - 1)
    xs, ys = [], []
    for k in range(n_bins):
        mask = (bin_ids == k)
        if mask.sum() > 0:
            xs.append(y_prob[mask].mean())
            ys.append(y_true[mask].mean())
    return np.array(xs), np.array(ys)


# Créer la figure avec 2 panels
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), sharey=True)

for ax, prompt in zip(axes, ['A', 'B']):
    # Diagonale de calibration parfaite
    ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1.2,
            label='Perfect calibration', alpha=0.6)

    # Une courbe par modèle
    for model_key, config in MODELS_CONFIG.items():
        fichier = f"{OUTPUT}/{model_key}_{prompt}_predictions.parquet"
        if not os.path.exists(fichier):
            continue
        df = pd.read_parquet(fichier)
        df['correct'] = (df['pred'] == df['expert']).astype(int)
        xs, ys = compute_bins(df['correct'].values, df['confidence'].values)
        ax.plot(xs, ys, 'o-', color=config['color'], linewidth=2,
                markersize=6, label=config['label'], alpha=0.9)

    # Mise en forme
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Predicted confidence')
    ax.set_title(f"Prompt {prompt} " +
                 ('(minimal)' if prompt == 'A' else '(enriched)'),
                 fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle=':')
    ax.set_aspect('equal')

# Y label uniquement à gauche
axes[0].set_ylabel('Observed accuracy')

# Légende commune en bas
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.05), frameon=True, fancybox=False,
           edgecolor='black')

plt.tight_layout()
plt.subplots_adjust(bottom=0.15)

# Sauvegarder en PDF (vectoriel, meilleur pour LaTeX)
plt.savefig(f"{OUTPUT}/reliability_diagram_publication.pdf",
            bbox_inches='tight', dpi=300)
plt.savefig(f"{OUTPUT}/reliability_diagram_publication.png",
            bbox_inches='tight', dpi=200)
plt.show()

print(f"\n✓ Sauvegardé : reliability_diagram_publication.pdf et .png")

In [3]:
import sys, torch, transformers, sklearn, numpy, pandas, scipy, matplotlib
print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"numpy : {numpy.__version__}")
print(f"pandas : {pandas.__version__}")
print(f"scipy : {scipy.__version__}")
print(f"matplotlib : {matplotlib.__version__}")

Python : 3.12.13
PyTorch : 2.10.0+cu128
transformers : 5.0.0
scikit-learn : 1.6.1
numpy : 2.4.6
pandas : 2.3.3
scipy : 1.16.3
matplotlib : 3.10.0
